# Conformal 2D meshing with `confMesh2dGMSH`

Canonical demo for the **raw Gmsh** 2D conformal path: mesh, grain-coloured plot, boundary NSETs, Abaqus `.inp` export.

The pygmsh class `confMesh2d` is deprecated. Requires `gmsh` (`pip install gmsh` or `pip install upxo[mesh]`).

In [ ]:
from pathlib import Path
from shapely.geometry import box, MultiPolygon

from upxo.meshing.conformal_mesher2d import confMesh2dGMSH
from upxo.meshing.gsmesh2d import mesh_gs, visualize_gs_mesh

## 1. Grain polygons

Any `{grain_id: Shapely Polygon or MultiPolygon}` works — Voronoi, polygonised MCGS, or a hand-built RVE as below.

In [ ]:
cells = {
    1: box(0, 0, 2, 2),
    2: box(2, 0, 4, 2),
    3: box(0, 2, 2, 4),
    4: box(2, 2, 4, 4),
    5: MultiPolygon([box(4, 0, 5, 1), box(4, 3, 5, 4)]),  # two parts, one grain
}
print(len(cells), 'grains')

## 2. Mesh (`mesh_gs`)

In [ ]:
result = mesh_gs(
    cells,
    method='conformal',
    mesh_size_gb=0.35,
    mesh_size_bulk=0.7,
    mesh_algo=6,
    recombine_to_quads=False,
    verbose=True,
)
m = result['mesher']
print(result['n_tri'], 'triangles,', result['n_nodes'], 'nodes')
print(m.validation_report)

## 3. ELSETs and NSETs

In [ ]:
m.form_elsets_gmsh()
m.build_boundary_nsets()
m.build_grain_nsets()
m.build_gb_nset()
print('ELSETs:', sorted(m.elsets))
print('NSET sizes:', {k: len(v) for k, v in m.nsets.items() if k.startswith('NS') or k in ('LEFT','RIGHT','TOP','BOTTOM','GB')})

## 4. Visualisation

Grain fill + GB overlay. `show_nsets=True` marks LEFT/RIGHT/TOP/BOTTOM nodes for BC setup.

In [ ]:
fig, ax = visualize_gs_mesh(result, figsize=(6, 6), dpi=130, show_nsets=True)
fig

In [ ]:
fig2, ax2 = m.plot_by_grain(figsize=(6, 6), show_gb=True, show_nsets=False,
                            title='Elements coloured by grain ELSET')
fig2

## 5. Export to Abaqus

`export_abaqus_inp` writes `*Node`, `*Element` (CPS3/CPS4 or CPE3/CPE4), per-grain `*Elset`, face `*Nset` (`NS_LEFT`, …, `NS_GB`), and dummy `*Solid Section` / `*Material` blocks (edit moduli before a real job).

Gmsh native `.msh` / `.vtk` remain available via `femesh_gmsh(..., formats=['msh','vtk'], out_dir=...)`.

In [ ]:
from upxo.meshing.writer_ABQ import summarize_inp
out_dir = Path.cwd() / 'confMesh2d_gmsh_out'
inp = m.export_abaqus_inp(out_dir / 'rve_cps3.inp', plane='stress')
inp, summarize_inp(inp)

## 6. `from_geometric_pxtal` (drop-in for old pygmsh notebooks)

In [ ]:
pxtal = MultiPolygon([box(0, 0, 1, 1), box(1, 0, 2, 1), box(0, 1, 1, 2)])
m2 = confMesh2dGMSH.from_geometric_pxtal(
    pxtal=pxtal, xbound=(0, 2), ybound=(0, 2))
m2.femesh_gmsh(mesh_size_gb=0.3, mesh_size_bulk=0.6, recombine_to_quads=False)
m2.form_elsets_gmsh()
P, L, T, Q = m2.get_mesh_geometry()
print('triangles', 0 if T is None else len(T), 'elsets', sorted(m2.elsets))
fig3, _ = m2.plot_by_grain(figsize=(5, 5), title='from_geometric_pxtal')
fig3